# Analyse et modélisation

Ce notebook présente la version finale du travail réalisée dans l'application Streamlit : exploration, préparation des données, entraînement, évaluation et ciblage marketing.

In [10]:
import importlib
import pandas as pd
from sklearn.metrics import classification_report, f1_score, roc_auc_score, average_precision_score

import fonctions
importlib.reload(fonctions)

clients_outliers_summary = fonctions.clients_outliers_summary
clients_marketing_profile_summary = fonctions.clients_marketing_profile_summary
entrainer_modele_rf = fonctions.entrainer_modele_rf
generer_predictions_marketing = fonctions.generer_predictions_marketing
preparer_dataset_complet = fonctions.preparer_dataset_complet

In [11]:
train_info = pd.read_csv('Data/train_info.csv')
clients_a_contacter = pd.read_csv('Data/clients_a_contacter.csv')

train_info.shape, clients_a_contacter.shape

((381109, 12), (127037, 11))

## 1. Contrôle rapide du dataset

Le jeu d'entraînement contient 381109 lignes et 12 colonnes. Le fichier de production contient 127037 clients à scorer.

In [12]:
train_info.dtypes

id_client                int64
genre                   object
age                      int64
permis_conduire          int64
code_regional          float64
ancien_assure            int64
age_vehicule            object
vehicule_endommage      object
prime_annuelle         float64
canal_communication    float64
anciennete               int64
reponse_client           int64
dtype: object

In [13]:
train_info.isna().sum().sort_values(ascending=False)

id_client              0
genre                  0
age                    0
permis_conduire        0
code_regional          0
ancien_assure          0
age_vehicule           0
vehicule_endommage     0
prime_annuelle         0
canal_communication    0
anciennete             0
reponse_client         0
dtype: int64

In [14]:
train_info['reponse_client'].value_counts(normalize=True)

reponse_client
0    0.877437
1    0.122563
Name: proportion, dtype: float64

## 2. Valeurs aberrantes

L'analyse par IQR permet d'expliciter la présence d'observations extrêmes. La prime annuelle présente quelques valeurs hautes, mais leur part reste limitée. Dans l'application, elles sont conservées et gérées par `RobustScaler`.

In [15]:
clients_outliers_summary(train_info, ['age', 'prime_annuelle', 'anciennete'])

,variable,q1,q3,iqr,borne_basse,borne_haute,nb_outliers,part_outliers_pct,conclusion_metier
0,age,25.0,49.0,24.0,-11.0,85.0,0,0.00,Impact limité
1,prime_annuelle,24405.0,39400.0,14995.0,1912.5,61892.5,10320,2.71,Impact limité
2,anciennete,82.0,227.0,145.0,-135.5,444.5,0,0.00,Impact limité


## 3. Préparation des données

Le pipeline final suit la logique retenue dans `app.py` :
- `OneHotEncoder` pour `genre` et `vehicule_endommage`
- `OrdinalEncoder` pour `age_vehicule`
- encodage métier pour `code_regional` et `canal_communication`
- création de `tranche_age` et de variables d'interaction

In [16]:
df_model_prep, preprocessing_artifacts = preparer_dataset_complet(train_info, is_train=True)
df_model_prep.columns.tolist()

['age',
 'permis_conduire',
 'ancien_assure',
 'age_vehicule',
 'prime_annuelle',
 'anciennete',
 'reponse_client',
 'tranche_age',
 'code_regional_score',
 'canal_communication_score',
 'genre_femelle',
 'genre_male',
 'vehicule_endommage_no',
 'vehicule_endommage_oui',
 'inter_age_dommage',
 'inter_age_ancien_assure',
 'inter_vehicule_ancien']

## 4. Entraînement, tuning et évaluation

Le modèle final est une forêt aléatoire avec une recherche légère d'hyperparamètres. Les métriques importantes sont F1, ROC-AUC et PR-AUC, car la classe positive est minoritaire.

In [17]:
model, scaler, X_test, y_test, feature_names, best_params, best_cv_f1 = entrainer_modele_rf(df_model_prep)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [9]:
print(classification_report(y_test, y_pred))
print('F1-score     :', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC      :', round(roc_auc_score(y_test, y_proba), 4))
print('PR-AUC       :', round(average_precision_score(y_test, y_proba), 4))
print('Meilleurs paramètres :', best_params)
print('Meilleur F1 CV :', round(best_cv_f1, 4))

              precision    recall  f1-score   support

           0       0.98      0.68      0.81     66880
           1       0.29      0.92      0.44      9342

    accuracy                           0.71     76222
   macro avg       0.64      0.80      0.62     76222
weighted avg       0.90      0.71      0.76     76222

F1-score     : 0.439
ROC-AUC      : 0.8568
PR-AUC       : 0.3675
Meilleurs paramètres : {'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 150}
Meilleur F1 CV : 0.4398


## 5. Ciblage marketing final

Les prédictions sur `clients_a_contacter.csv` sont converties en stratégie métier. La priorité est la zone d'influence, c'est-à-dire les clients qu'une action commerciale peut encore faire basculer.

In [18]:
liste_finale = generer_predictions_marketing(
    model,
    scaler,
    clients_a_contacter,
    feature_names,
    preprocessing_artifacts,
)
liste_finale['strategie_marketing'].value_counts()

strategie_marketing
Peu probable (Ne pas contacter)      65619
Secondaire                           34690
À CIBLER (Zone d'influence)          19283
Presque certain (Contact inutile)     7445
Name: count, dtype: int64

In [19]:
cibles = liste_finale[liste_finale['strategie_marketing'].str.contains('CIBLER', na=False)]
clients_marketing_profile_summary(cibles)

,indicateur,valeur
0,Nombre de clients ciblés,19283
1,Âge moyen,43.5
2,Prime annuelle moyenne,30040.2
3,Ancienneté moyenne,155.4
4,Genre majoritaire,male
5,Âge véhicule majoritaire,< 1 an
6,Véhicule endommagé majoritaire,oui


## Conclusion

Le pipeline final du notebook est aligné sur l'application Streamlit. Il justifie explicitement les choix de préparation, de modélisation, d'évaluation et de ciblage marketing utilisés dans le rendu final.